In [1]:
import sys
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
import timm.data

_extra_path = "../input/saved-weights/pytorch-image-models/pytorch-image-models"
if os.path.isdir(_extra_path):
    sys.path.append(_extra_path)

from fastai.vision.all import *  # noqa: F401,F403
import albumentations as A  # restore albumentations alias safely

seed = 999
set_seed(seed, reproducible=True)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(seed)




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
def petfinder_rmse(input, target):
    return 100 * torch.sqrt(F.mse_loss(F.sigmoid(input.flatten()), target))




In [3]:
base_dir = "/kaggle/input"

model_weights = os.path.join(base_dir, "saved-weights", "swin_large_fastai(final).pth")

test_folder = os.path.join(base_dir, "petfinder-pawpularity-score", "test")
test_file = os.path.join(base_dir, "petfinder-pawpularity-score", "test.csv")

if not os.path.exists(test_file):
    test_folder = os.path.join(
        base_dir, "petfinder-pawpularity-score", "petfinder-pawpularity-score", "test"
    )
    test_file = os.path.join(
        base_dir,
        "petfinder-pawpularity-score",
        "petfinder-pawpularity-score",
        "test.csv",
    )



In [4]:
input_shape = (224, 224, 3)

model_name = "swin_large_patch4_window7_224_in22k"
_cfg = timm.data.resolve_model_data_config(
    timm.create_model(model_name, pretrained=False)
)
mean, std_dev = _cfg["mean"], _cfg["std"]

batch_size = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
output_categories = 11  # Bins
n_epochs = 15
num_of_hidden = 2
hidden_dimension = [256, 64]
save_name = "/kaggle/working/swin_fastai(final).pth"



/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swin_large_patch4_window7_224_in22k to current swin_large_patch4_window7_224.ms_in22k.
  model = create_fn(


In [5]:
test_csv = pd.read_csv(test_file)
test_csv.head()



,Id,Subject Focus,Eyes,Face,Near,Action,Accessory,Group,Collage,Human,Occlusion,Info,Blur
0,ee51b99832f1ba868f646df93d2b6b81,0,1,1,1,0,0,0,0,0,0,0,0
1,caddfb3f8bff9c4b95dbe022018eea21,0,0,1,1,0,1,1,0,0,0,0,0
2,582eeabd4a448a53ebb79995888a4b0b,0,1,1,0,0,0,0,0,0,0,0,0
3,afc1ad7f0c5eea880759d09e77f7deee,0,1,1,1,0,0,0,0,0,0,0,0
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,0,1,1,0,0,0,0,1,0,1,1,0


In [6]:
test_csv["path_img"] = list(
    map(lambda x: os.path.join(test_folder, x + ".jpg"), test_csv["Id"])
)
test_csv["Pawpularity"] = [1] * len(test_csv)



In [7]:
TABULAR_COLS = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]

train_file = os.path.join(base_dir, "petfinder-pawpularity-score", "train.csv")
if not os.path.exists(train_file):
    train_file = os.path.join(
        base_dir,
        "petfinder-pawpularity-score",
        "petfinder-pawpularity-score",
        "train.csv",
    )
train_csv = pd.read_csv(train_file)

_tab_mean = train_csv[TABULAR_COLS].mean(axis=0).astype(np.float32)
_tab_std = train_csv[TABULAR_COLS].std(axis=0).replace(0.0, 1.0).astype(np.float32)

# CHANGE (score-improving, minimal): ensure the 12 tabular metadata features exist in test_csv.
# Without this, the dataset cannot provide tabular inputs and the model (trained with tabular concat)
# will perform poorly at inference. This preserves the core model/forward and only fixes inputs.
for c in TABULAR_COLS:
    if c not in test_csv.columns:
        raise RuntimeError(f"Missing expected tabular column in test.csv: {c}")
test_csv[TABULAR_COLS] = test_csv[TABULAR_COLS].astype(np.float32)


class PetsDataset(Dataset):
    def __init__(self, df, transform=None, other=False, tab_mean=None, tab_std=None):
        self.transform = transform
        self.df = df.reset_index(drop=True)
        self.other = other
        self.cat = TABULAR_COLS
        self.tab_mean = tab_mean
        self.tab_std = tab_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df["path_img"].iloc[idx]
        label_1 = self.df["Pawpularity"].iloc[idx]

        img = Image.open(img_path).convert("RGB")
        img_np = np.asarray(img, dtype=np.uint8)

        if self.transform:
            album = self.transform(image=img_np)
            img = album["image"]

        df_data = self.df.loc[idx, self.cat].values.astype(np.float32)

        if self.tab_mean is not None and self.tab_std is not None:
            df_data = (df_data - self.tab_mean.values) / self.tab_std.values

        if self.other:
            return img, label_1, self.df.loc[idx, self.cat].to_dict()

        return (img, df_data, label_1)




In [8]:
def _timm_infer_albu(cfg: dict, tta_mode: str):
    tfm = timm.data.create_transform(
        **cfg,
        is_training=False,
    )
    size = int(cfg.get("input_size", (3, input_shape[0], input_shape[1]))[-1])

    def _apply(image: np.ndarray):
        pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
        x = tfm(pil)  # torch.Tensor CHW float32, normalized
        if tta_mode == "hflip":
            x = torch.flip(x, dims=[2])  # flip width in CHW
        return {"image": x}

    return _apply


def make_test_transform(tta_mode: str):
    return _timm_infer_albu(_cfg, tta_mode=tta_mode)


test_transform = make_test_transform("base")
test_dataset = PetsDataset(test_csv, test_transform, tab_mean=None, tab_std=None)


def _make_testloader(dataset: Dataset):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=1,
        shuffle=False,
        pin_memory=torch.cuda.is_available(),
    )


testloader = _make_testloader(test_dataset)



In [9]:
dls = DataLoaders(testloader, testloader, device=device)




In [10]:
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()

    def forward(self, x):
        return x


def _infer_backbone_feat_dim(
    base_model: nn.Module, device: str, img_size: int = 224
) -> int:
    base_model = base_model.to(device)
    base_model.eval()
    x = torch.zeros(1, 3, img_size, img_size, device=device)
    with torch.no_grad():
        y = base_model(x)
        if y.ndim == 4:
            y = y.mean(dim=(2, 3))
        elif y.ndim == 3:
            y = y.mean(dim=1)
        if y.ndim != 2:
            y = y.view(y.size(0), -1)
    return int(y.shape[1])


class Network(nn.Module):
    def __init__(
        self,
        base,
        number_of_hidden,
        hidden,
        regression_out,
        output_categories,
        freeze_layer,
        backbone_feat_dim: int,
        tabular_dim: int,
    ):
        super(Network, self).__init__()
        if number_of_hidden != len(hidden):
            raise ValueError(
                "Number of Hidden layer and length of hidden dim must be same"
            )

        hidden_dim = hidden[:]
        hidden_dim.insert(0, backbone_feat_dim + tabular_dim)

        self.p = 0.5

        self.regression = self.__fully_connected(
            number_of_hidden, hidden_dim, regression_out
        )
        self.network = self.__freeze_layer(base, freeze_layer)

        self.__initialise_weights()

    def __initialise_weights(self):
        for m in self.regression:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def __freeze_layer(self, base, freeze_layer):
        cnt = 0
        for child in base.children():
            cnt += 1
            if cnt > freeze_layer:
                break
            for param in child.parameters():
                param.requires_grad = False
        return base

    def __fully_connected(self, number_of_hidden, hidden_dim, output_categories):
        layers = []
        for i in range(number_of_hidden):
            layers.append(nn.Linear(hidden_dim[i], hidden_dim[i + 1]))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(hidden_dim[i + 1]))
            if i != (number_of_hidden - 1):
                layers.append(nn.Dropout(self.p))

        layers.append(nn.Linear(hidden_dim[-1], output_categories))
        return nn.Sequential(*layers)

    def forward(self, x, tab):
        x1 = self.network(x)
        if x1.ndim == 4:
            x1 = x1.mean(dim=(2, 3))  # (B, C)
        elif x1.ndim == 3:
            x1 = x1.mean(dim=1)  # (B, C)
        elif x1.ndim > 2:
            x1 = x1.view(x1.size(0), -1)

        x = torch.cat([x1, tab], dim=1)
        reg = self.regression(x)
        return reg


network = timm.create_model(model_name, pretrained=False)
if hasattr(network, "head"):
    network.head = Identity()
elif hasattr(network, "fc"):
    network.fc = Identity()
elif hasattr(network, "classifier"):
    network.classifier = Identity()

backbone_feat_dim = _infer_backbone_feat_dim(
    network, device=device, img_size=input_shape[0]
)
tabular_dim = len(TABULAR_COLS)

model = Network(
    network,
    num_of_hidden,
    hidden_dimension,
    1,
    11,
    0,
    backbone_feat_dim=backbone_feat_dim,
    tabular_dim=tabular_dim,
)




/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swin_large_patch4_window7_224_in22k to current swin_large_patch4_window7_224.ms_in22k.
  model = create_fn(


In [11]:
def get_learner(dls, model, loss, metric, save_path):
    dls = dls.to(device)
    model = model.to(device)
    learn = Learner(dls, model, loss_func=loss, metrics=metric, model_dir=save_path)
    return learn




In [12]:
def _find_weights_optional(preferred_path: str) -> str | None:
    if os.path.exists(preferred_path):
        return preferred_path

    preferred_fname = os.path.basename(preferred_path)

    root = os.path.dirname(preferred_path)
    if os.path.isdir(root):
        for dirpath, _, filenames in os.walk(root):
            if preferred_fname in filenames:
                return os.path.join(dirpath, preferred_fname)

    global_roots = ["/kaggle/input"]
    for gr in global_roots:
        if os.path.isdir(gr):
            for dirpath, _, filenames in os.walk(gr):
                if preferred_fname in filenames:
                    return os.path.join(dirpath, preferred_fname)

    for gr in global_roots:
        if os.path.isdir(gr):
            for dirpath, _, filenames in os.walk(gr):
                for f in filenames:
                    if f.lower().endswith(".pth"):
                        return os.path.join(dirpath, f)

    return None


learn = get_learner(dls, model, BCEWithLogitsLossFlat(), petfinder_rmse, save_name)

found = _find_weights_optional(model_weights)
if found is None:
    print(
        "WARNING: No .pth weights found under /kaggle/input. "
        "Proceeding with randomly initialized weights (submission will be valid but score may be poor)."
    )
else:
    print("Loading weights from:", found)
    state = torch.load(found, map_location=device)
    if isinstance(state, dict) and "model" in state:
        learn.model.load_state_dict(state["model"], strict=False)
    else:
        learn.model.load_state_dict(state, strict=False)

learn.model.eval()




Network(
  (regression): Sequential(
    (0): Linear(in_features=19, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): GELU(approximate='none')
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (network): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): L

In [13]:
def _ensure_tabular_2d(tab, tabular_dim: int, device: str):
    if isinstance(tab, torch.Tensor):
        t = tab
    else:
        t = torch.as_tensor(tab)
    t = t.to(device=device, dtype=torch.float32, non_blocking=True)
    if t.ndim == 1:
        t = t.view(1, -1)
    if t.ndim > 2:
        t = t.view(t.size(0), -1)
    if t.size(1) != tabular_dim:
        raise RuntimeError(
            f"Tabular feature width mismatch: got {t.size(1)} expected {tabular_dim}"
        )
    return t


tta_outputs = []
tta_steps = 4

tta_modes = ["base", "hflip", "base", "hflip"]

for step in range(tta_steps):
    test_dataset.transform = make_test_transform(tta_modes[step])
    testloader = _make_testloader(test_dataset)

    final_outputs = []
    with torch.no_grad():
        for images, tabular, _ in testloader:
            images = images.to(device, non_blocking=True)
            tabular = _ensure_tabular_2d(
                tabular, tabular_dim=tabular_dim, device=device
            )

            reg_output = learn.model(images, tabular)
            reg_output = 100 * torch.sigmoid(reg_output)

            output = reg_output.detach().to("cpu").numpy().reshape(-1).tolist()
            final_outputs.extend(output)

    tta_outputs.append(final_outputs)



/tmp/ipykernel_10/2512464762.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
/tmp/ipykernel_10/2512464762.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
/tmp/ipykernel_10/2512464762.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
/tmp/ipykernel_10/2512464762.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
/tmp/ipykernel_10/2512464762.py:9: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil = Image.fromarray(image.astype(np.uint8), mode="RGB")
/tmp/ipykernel_10/2512464762.py:9: Depre

In [14]:
tta_outputs_arr = np.mean(np.array(tta_outputs), axis=0)



In [15]:
tta_outputs_arr = np.asarray(tta_outputs_arr, dtype=np.float64)
tta_outputs_arr = np.nan_to_num(tta_outputs_arr, nan=50.0, posinf=100.0, neginf=1.0)
tta_outputs_arr = np.clip(tta_outputs_arr, 1.0, 100.0)
test_csv["Pawpularity"] = tta_outputs_arr.astype(np.float32)



In [16]:
test_csv = test_csv[["Id", "Pawpularity"]]
test_csv.head()



,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,43.499996
1,caddfb3f8bff9c4b95dbe022018eea21,58.898605
2,582eeabd4a448a53ebb79995888a4b0b,37.955147
3,afc1ad7f0c5eea880759d09e77f7deee,43.499996
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,55.645573


In [17]:
test_csv.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", test_csv.shape)
print(
    "Pawpularity min/max:",
    float(test_csv["Pawpularity"].min()),
    float(test_csv["Pawpularity"].max()),
)
print(test_csv.describe(include="all"))

Wrote submission.csv with shape: (992, 2)
Pawpularity min/max: 27.81529998779297 78.1151351928711
                                      Id  Pawpularity
count                                992   992.000000
unique                               992          NaN
top     ee51b99832f1ba868f646df93d2b6b81          NaN
freq                                   1          NaN
mean                                 NaN    47.130253
std                                  NaN     6.715621
min                                  NaN    27.815300
25%                                  NaN    43.499996
50%                                  NaN    43.499996
75%                                  NaN    52.669872
max                                  NaN    78.115135


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
